# 03d - Training the Comparison Models for the Leakage-Detection Protocol

**Step 18a. This notebook does not access the test set.**

`03_train.ipynb` trained three LoRA adapters, all fit on `clean/train` and differing only in
LoRA rank. This notebook trains the two remaining models required by the leakage-detection
protocol: one fine-tuned on `naive/train`, and one on `naive_sub/train`. Together with the
clean-split models, these constitute the leakage experiment - without them, any observed
leakage could only be characterised as a proportion of near-duplicate rows between splits,
rather than as a measurable difference in downstream model performance.

Both models are trained under the LoRA configuration frozen in `03b_resolution_floor.ipynb`,
holding the configuration constant so that any difference in outcome can be attributed to the
training data rather than to the model setup.

Training and evaluation are separated across notebooks and sessions by design: evaluation on
the held-out test set is performed exclusively in `04_test.ipynb`. This notebook has no code
path to `clean/test` or `naive/test`, and an automated check confirms the absence of any
reference to these files. This separation guarantees that the test set is accessed exactly
once, at the final evaluation stage, preserving the validity of the evaluation and eliminating
any risk of information leakage from repeated or exploratory access.

Results are persisted after each model completes, so that an interrupted session affects only
the model currently in progress.

## 0 - Environment Setup and Prerequisite Checks

This section mirrors the setup used in `03_train.ipynb` and `03b_resolution_floor.ipynb` to
ensure consistency across the protocol. On the Colab runtime, notebook cells execute from the
local editor session, while the `src/` module code is obtained from a freshly cloned copy of
the repository. Consequently, only code that has been committed and pushed to the repository
is executed; local, unpushed modifications to `src/protocol_models.py` have no effect on this
run.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Emma-V/support-triage.git"
BRANCH   = "main"
CLONE_TO = Path("/content/support-triage")


def _git(*args, cwd=None) -> str:
    return subprocess.run(["git", *args], cwd=cwd, check=True,
                          capture_output=True, text=True).stdout.strip()


def _find_repo(start: Path):
    here = start.resolve()
    while not (here / "src" / "data.py").exists():
        if here == here.parent:
            return None
        here = here.parent
    return here


REPO_ROOT = _find_repo(Path.cwd())

if REPO_ROOT is None or REPO_ROOT == CLONE_TO:
    if (CLONE_TO / ".git").exists():
        _git("fetch", "origin", BRANCH, cwd=CLONE_TO)
        _git("reset", "--hard", f"origin/{BRANCH}", cwd=CLONE_TO)
        print(f"updated the existing clone at {CLONE_TO}")
    else:
        _git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(CLONE_TO))
        print(f"cloned {REPO_URL} to {CLONE_TO}")
    REPO_ROOT = CLONE_TO

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print(f"\nrepo   {REPO_ROOT}")
print(f"commit {_git('rev-parse', '--short', 'HEAD', cwd=REPO_ROOT)}  "
      f"{_git('log', '-1', '--pretty=%s', cwd=REPO_ROOT)}")
print("\n^ src/protocol_models.py is required here, and src/train.py needs")
print("  trained_on / scored_on. If this commit predates them the imports below")
print("  fail - and worse, a record would name the wrong split. Push first.")

cloned https://github.com/Emma-V/support-triage.git to /content/support-triage

repo   /content/support-triage
commit dcd3357  Add confidence and calibration analysis for the frozen r=8 model

^ src/protocol_models.py is required here, and src/train.py needs
  trained_on / scored_on. If this commit predates them the imports below
  fail - and worse, a record would name the wrong split. Push first.


In [2]:
# The same two pinned packages as in 03_train.ipynb, and the same stale torchao removal.
# Unchanged on purpose: a different library stack would make these runs
# incomparable to the earlier ones, which is the one thing this notebook cannot afford.
import importlib
import importlib.metadata as metadata
import subprocess
import sys

PINNED = {"transformers": "5.14.1", "peft": "0.20.0"}
TORCHAO_MIN_FOR_PEFT = (0, 16)

try:
    have_torchao = metadata.version("torchao")
except metadata.PackageNotFoundError:
    have_torchao = None

stale_torchao = have_torchao is not None and tuple(
    int("".join(c for c in chunk if c.isdigit()) or "0")
    for chunk in have_torchao.split(".")[:2]
) < TORCHAO_MIN_FOR_PEFT

print(f"{'torchao':14s} found {have_torchao or 'nothing':10s} "
      f"{'too old for peft - removing it' if stale_torchao else 'not in the way'}")
if stale_torchao:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=True)
    importlib.invalidate_caches()

replaced = []
for package, want in PINNED.items():
    try:
        have = metadata.version(package)
    except metadata.PackageNotFoundError:
        have = None
    print(f"{package:14s} found {have or 'nothing':10s} want {want}")
    if have != want:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"{package}=={want}"], check=True)
        replaced.append(f"{package} {have or 'missing'} -> {want}")

if replaced:
    print("\n" + "!" * 70)
    print("REPLACED: " + "; ".join(replaced))
    loaded = [name for name in PINNED if name in sys.modules]
    if loaded:
        print(f"Python is already holding {', '.join(loaded)} in memory. RESTART THE")
        print("KERNEL and run from the top - everything above here is cheap.")
        print("!" * 70)
        raise RuntimeError("restart the kernel: a pinned package was replaced under it")
    print("!" * 70)

import peft
import torch
import transformers

print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("\n!! No GPU on this runtime. Reconnect and pick a T4 runtime.")

torchao        found 0.10.0     too old for peft - removing it
transformers   found 5.15.1     want 5.14.1
peft           found 0.20.0     want 0.20.0

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
REPLACED: transformers 5.15.1 -> 5.14.1
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
torch 2.11.0+cu128 | transformers 5.14.1 | peft 0.20.0
CUDA available: True


In [3]:
import importlib.util
import gc, json, os, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

# Colab is detected by the module, not by /content. On Windows a leading slash is
# drive-relative - Path("/content") is C:\content - so a local run that once fell
# back to /content/_local_runs creates the very directory that makes every later
# local run claim to be Colab. google.colab is what the mount actually needs.
IN_COLAB = importlib.util.find_spec("google.colab") is not None

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "data.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "data.py").exists(), (
    f"No repository found above {Path.cwd()}. Run the bootstrap cell above first - "
    "on a Colab runtime it is what puts the repo on the machine.")
sys.path.insert(0, str(REPO_ROOT))

from src import data as D
from src import evaluate as E
from src import protocol_models as P
from src import train as T

METRICS_DIR = REPO_ROOT / "results" / "metrics"
ERRORS_DIR  = REPO_ROOT / "results" / "errors"
ARTIFACTS   = REPO_ROOT / "artifacts"
for _d in (METRICS_DIR, ERRORS_DIR, ARTIFACTS):
    _d.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT.name, "| colab:", IN_COLAB)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

# The staleness guard. On Colab these cells come from the editor and src/ comes
# from the clone, so an unpushed edit is not here - and without this check that
# mismatch surfaces at the first call site, which is AFTER the first model has
# finished training. Eighty minutes of GPU time to discover a missing function is
# not an acceptable way to find out.
_REQUIRED = {
    P: ["TRAINING_PLAN", "run_name", "assert_matches_freeze", "matches_freeze",
        "differences_from_freeze", "score_from_logits", "save_val_outputs"],
    D: ["load_naive_sub", "SUBSAMPLE_SEED", "load_split", "sha256_of_split"],
    T: ["train_one_run", "load_adapter", "encode_split", "predict_logits", "gpu_report"],
}
_missing = {m.__name__: [n for n in names if not hasattr(m, n)]
            for m, names in _REQUIRED.items()}
_missing = {mod: names for mod, names in _missing.items() if names}

# RunConfig gaining trained_on is checked separately: it is an attribute of the
# CLASS, not of the module, so the loop above cannot see it. Without it every
# record below claims to have trained on clean/train - and a record naming the
# wrong split does not look wrong, it looks like a different finding.
if not hasattr(T.RunConfig(name="probe", r=1), "trained_on"):
    _missing["src.train.RunConfig"] = ["trained_on", "scored_on", "subsample_seed"]

if _missing:
    print("!" * 70)
    for mod, names in _missing.items():
        print(f"{mod} is missing: {', '.join(names)}")
    print()
    print("The src/ on this runtime is OLDER than this notebook.")
    print("  1. push the current src/ from your machine")
    print("  2. RESTART THE KERNEL - re-running is not enough, Python caches an")
    print("     imported module in sys.modules and an import will not re-read it")
    print("  3. run this notebook from the top")
    print("!" * 70)
    raise ImportError(f"stale src/ on this runtime: {_missing}")

print("src/ has every function and field this notebook uses")

repo: support-triage | colab: True
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3
src/ has every function and field this notebook uses


### Persistent Storage Verification

Because the Colab runtime is ephemeral, all results must be mirrored to persistent storage
(Google Drive) during the session to avoid loss upon disconnection. This notebook therefore
verifies that Drive is mounted before training begins, and raises an error rather than
proceeding if it is not, since continuing without persistent storage risks the irrecoverable
loss of trained models and metrics.

In [4]:
for _n in ("IN_COLAB", "REPO_ROOT"):
    if _n not in globals():
        raise RuntimeError(
            f"{_n} is not defined - run the imports-and-paths cell above first (the one "
            "that prints: repo ... | colab ...). If that cell is the one that failed, fix "
            "it there: its error is the real one.")

# True ONLY to rehearse on a runtime where Drive cannot mount, accepting that
# everything the run produces dies with the machine. A variable rather than a
# comment, so "I meant to run without Drive" is a recorded choice in the notebook
# instead of a warning somebody scrolled past ninety minutes earlier.
ALLOW_NO_DRIVE = False

DRIVE_OK = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = Path("/content/drive/MyDrive").exists()
    except Exception as exc:
        print(f"!! Drive did not mount - {type(exc).__name__}: {exc}")

if IN_COLAB and not DRIVE_OK and not ALLOW_NO_DRIVE:
    raise RuntimeError(
        "Drive is not mounted, so nothing this notebook produces would survive the "
        "runtime - and today that is two models and about 2.2 hours. Fix it before "
        "the training starts:\n"
        "  - re-run this cell: the mount is flaky and often works the second time\n"
        "  - update the Colab VS Code extension (drive.mount needs v0.2.1+), or\n"
        "  - run this notebook in the Colab web UI instead.\n"
        "To rehearse without Drive anyway, set ALLOW_NO_DRIVE = True above and accept "
        "that every output below is disposable.")

if IN_COLAB and not DRIVE_OK:
    print("\n" + "!" * 70)
    print("RUNNING WITHOUT DRIVE, deliberately (ALLOW_NO_DRIVE = True). Adapters and")
    print("run records go to the runtime's own disk and are DELETED when it")
    print("disconnects. Nothing produced below is safe to report.")
    print("!" * 70)

DRIVE_ROOT = (Path("/content/drive/MyDrive/support-triage") if DRIVE_OK
              else Path("/content/_local_runs") if IN_COLAB
              else REPO_ROOT / "_local_runs")
RUNS_DIR = DRIVE_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# The split CSVs are committed, so a clone arrives with them. Missing files here
# mean a clone older than the commit that added them - a one-line fix, and not a
# data problem, so it should not read like one.
PROCESSED = REPO_ROOT / "data" / "processed"
for _needed in ("clean/train.csv", "naive/train.csv", "naive_sub/train.csv"):
    if not (PROCESSED / _needed).exists():
        raise FileNotFoundError(
            f"{PROCESSED / _needed} is missing. It is in git, so this clone predates "
            "the commit that added it: re-run the bootstrap cell at the top of section "
            "0 - it hard-resets the clone to origin/main and brings the data with it.")

print("runs go to:", RUNS_DIR)

Mounted at /content/drive
runs go to: /content/drive/MyDrive/support-triage/runs


### Incremental Persistence After Each Model

To guard against loss of results from an interrupted runtime, results are persisted
programmatically after each model finishes training, rather than only at the end of the
notebook. Each result is written to the two locations available within the credential-free
Colab runtime: a local commit within the cloned repository, preserving a chronological record
of the run, and a mirrored copy on Google Drive, which persists independently of the runtime
instance.

The repository is public and is graded via GitHub, so no push credentials are stored within
the Colab environment. Pushing the committed results to the remote repository is therefore
performed as a separate restore step, from the local machine, using the author's own
credentials.

In [5]:
import shutil

# This project stores no push credential anywhere - the repo is public and is
# graded from GitHub, so nothing secret may ever live in a Colab runtime or in
# this notebook. The design is therefore: the clone commits locally (so the
# order of what happened stays recorded inside the session), Drive holds the
# mirror that survives the runtime, and the restore step on the local machine
# is what brings these files into git, pushed with Emma's normal credentials.
PERSIST_PATHS = ("results", "artifacts", "data/processed/naive_sub")


def _run(args):
    return subprocess.run(args, cwd=REPO_ROOT, capture_output=True, text=True)


def persist(message: str, mirror: bool = True) -> None:
    """Commit locally, mirror to Drive. Never raises.

    Never raising is deliberate: this is called between measurements that are
    expensive to repeat, so a failure has to be reported and stepped over
    rather than allowed to take the rest of the sequence down with it.
    """
    _run(["git", "add", *PERSIST_PATHS])
    if _run(["git", "diff", "--cached", "--quiet"]).returncode == 0:
        print("  [git  ] nothing new to commit")
    else:
        committed = _run(["git", "-c", "user.email=chagitvain02@gmail.com",
                          "-c", "user.name=Emma Vainshtein", "commit", "-m", message])
        print("  [git  ] committed (local to this runtime - Drive is what survives)"
              if committed.returncode == 0
              else f"  [git  ] COMMIT FAILED: {committed.stderr.strip()[:200]}")

    if mirror and DRIVE_OK:
        for folder in PERSIST_PATHS:
            source = REPO_ROOT / folder
            if not source.exists():
                continue
            try:
                shutil.copytree(source, DRIVE_ROOT / folder, dirs_exist_ok=True)
            except Exception as exc:
                print(f"  [drive] {folder} FAILED: {type(exc).__name__}: {exc}")
        n = sum(1 for p in (DRIVE_ROOT / "results").rglob("*") if p.is_file())
        print(f"  [drive] mirrored, results/ on Drive now holds {n} files")
    elif mirror and IN_COLAB:
        print("  [drive] NOT mirrored - Drive is not mounted, this is disposable")


persist("03d session opened")


  [git  ] nothing new to commit
  [drive] mirrored, results/ on Drive now holds 69 files


In [6]:
VERSIONS = T.check_transformers_version()
print("transformers", VERSIONS["transformers"], ">=", VERSIONS["min_required"], "OK")

HARDWARE = T.gpu_report()
for k, v in HARDWARE.items():
    print(f"  {k:22s} {v}")

if HARDWARE.get("device") != "cuda":
    raise RuntimeError(
        "No GPU on this runtime. Two 1.7B fine-tunes on CPU is not a slow run, it is a "
        "run that does not finish. Reconnect and pick a T4.")

print(f"\nPRECISION DECIDED: {HARDWARE['precision']} - {HARDWARE['precision_reason']}")

transformers 5.14.1 >= 4.51 OK
  gpu_name               NVIDIA L4
  compute_capability     8.9
  total_memory_gb        23.7
  bf16_supported         True
  device                 cuda
  precision              bf16
  precision_reason       bf16 supported by this GPU - preferred, no loss scaling needed

PRECISION DECIDED: bf16 - bf16 supported by this GPU - preferred, no loss scaling needed


## 1 - Entry Conditions

The protocol requires that the frozen configuration record exist and that the resolution floor
has been measured before the test set may be accessed at all, in any notebook. These conditions
are validated here as well, because the frozen configuration is the reference against which
both models trained in this notebook are compared.

The training and validation splits are loaded individually, by name, and no test-set object is
ever constructed in this notebook. The convenience function that loads all six splits at once,
`D.load_all_splits()`, is deliberately avoided, since it would also load the test data.

In [7]:
FREEZE_PATH = ARTIFACTS / "config_freeze.json"
if not FREEZE_PATH.exists():
    raise FileNotFoundError(
        "artifacts/config_freeze.json is missing. It is the entry condition for this "
        "stage, and it is also what the two runs below are checked against - without it "
        "a comparison between protocols could quietly also be a comparison between "
        "configurations. It is written by notebooks/03b_resolution_floor.ipynb; run "
        "that first and commit its output.")
FREEZE = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))

frozen_at = datetime.fromisoformat(FREEZE["frozen_at"])
now = datetime.now(timezone.utc)
assert frozen_at < now, "the freeze record is stamped in the future"
assert (now - frozen_at).total_seconds() > 3600, (
    f"the configuration was frozen {(now - frozen_at).total_seconds() / 60:.0f} minutes "
    "ago. A freeze written in the same session as the runs it governs is not a freeze.")

RESOLUTION = FREEZE["resolution_floor"]
FLOOR = RESOLUTION["floor"]
FROZEN_R = FREEZE["model"]["r"]
TRAIN_SEED = FREEZE["training"]["train_seed_of_frozen_run"]
assert FLOOR > 0, RESOLUTION

print(f"freeze written   {FREEZE['frozen_at']}  ({(now - frozen_at).days} days ago)")
print(f"frozen from      {FREEZE['frozen_from_run']}")
print(f"chosen r         {FROZEN_R}   (alpha {FREEZE['model']['lora_alpha']})")
print(f"train seed       {TRAIN_SEED}   - the only seed in this project")
print(f"resolution floor {FLOOR:.6f}   basis: {RESOLUTION['basis']}")
print(f"                 {RESOLUTION['limit']}")


freeze written   2026-08-27T14:51:35+00:00  (2 days ago)
frozen from      run_04_lora_r8
chosen r         8   (alpha 16)
train seed       42   - the only seed in this project
resolution floor 0.000477   basis: one_validation_row
                 this is metric resolution, not training variance - the project trains under one seed, so run-to-run spread was never measured and the true floor can only be larger


In [8]:
manifest = json.loads((PROCESSED / "split_manifest.json").read_text(encoding="utf-8"))

# Read one at a time and named one at a time. This is deliberately NOT
# D.load_all_splits(), which reads all six files - including the two this
# notebook must never touch.
TRAIN_FRAMES = {
    "clean/train":     D.load_split("clean", "train", PROCESSED),
    "naive/train":     D.load_split("naive", "train", PROCESSED),
    "naive_sub/train": D.load_naive_sub(PROCESSED),   # verifies its own sha256
}
VAL_FRAMES = {
    "clean/val": D.load_split("clean", "val", PROCESSED),
    "naive/val": D.load_split("naive", "val", PROCESSED),
}

# The manifest hashes cover the four committed files loaded above. It also holds
# hashes for the two test files, and verifying THOSE would mean reading them - so
# that check belongs to 04_test.ipynb, which is allowed to.
for name, frame in {**TRAIN_FRAMES, **VAL_FRAMES}.items():
    split, part = name.split("/")
    if split in manifest["splits"]:
        expected = manifest["splits"][split][part]["sha256"]
        actual = D.sha256_of_split(frame)
        assert actual == expected, (
            f"{name} does not match split_manifest.json:\n  on disk  {actual}\n"
            f"  manifest {expected}\nThese are not the rows every earlier number in "
            "this project was measured on.")

sub_manifest = json.loads((PROCESSED / "naive_sub" / "subsample_manifest.json"
                           ).read_text(encoding="utf-8"))
assert sub_manifest["subsample_seed"] == D.SUBSAMPLE_SEED
assert sub_manifest["drawn_from_sha256"] == manifest["splits"]["naive"]["train"]["sha256"], (
    "naive_sub was drawn from a different naive/train than the one on disk now")
assert len(TRAIN_FRAMES["naive_sub/train"]) == len(TRAIN_FRAMES["clean/train"]), (
    "the size control is not the size of the thing it is controlling for")

INTENTS = json.loads((ARTIFACTS / "labels.json").read_text(encoding="utf-8"))
assert len(INTENTS) == 27, len(INTENTS)

print("manifest verified - the four committed frames match their frozen fingerprints")
print(f"naive_sub verified - seed {sub_manifest['subsample_seed']}, "
      f"sha {sub_manifest['sha256'][:16]}, drawn from naive/train\n")
for name, frame in {**TRAIN_FRAMES, **VAL_FRAMES}.items():
    print(f"  {name:18s} {len(frame):>6,} rows")
print(f"  {'labels':18s} {len(INTENTS):>6,} intents, frozen order")
print("\nNo test frame was constructed. clean/test and naive/test were not opened,")
print("not hashed, and are not in this runtime's memory - that is 04_test.ipynb's job.")

manifest verified - the four committed frames match their frozen fingerprints
naive_sub verified - seed 42, sha d1a12470f92549af, drawn from naive/train

  clean/train         9,893 rows
  naive/train        17,187 rows
  naive_sub/train     9,893 rows
  clean/val           2,120 rows
  naive/val           3,683 rows
  labels                 27 intents, frozen order

No test frame was constructed. clean/test and naive/test were not opened,
not hashed, and are not in this runtime's memory - that is 04_test.ipynb's job.


## 2 - Runtime Budget Estimation

The expected runtime for each model is estimated from training times measured in
`03_train.ipynb`, rather than assumed in advance. This allows the feasibility of completing
both training runs within a single session to be assessed before training begins, rather than
discovered partway through.

In [9]:
sweep = pd.read_csv(METRICS_DIR / "lora_r_sweep.csv")
frozen_row = sweep.loc[sweep["r"] == FROZEN_R].iloc[0]
EPOCHS = int(FREEZE["training"]["epochs"])

# 03_train.ipynb trained 9,893 rows for 3 epochs at the frozen r. Everything below
# scales from that one measurement.
SECONDS_PER_ROW_EPOCH = float(frozen_row["runtime_seconds"]) / (
    len(TRAIN_FRAMES["clean/train"]) * EPOCHS)

rows, total = [], 0.0
for plan in P.TRAINING_PLAN:
    n = len(TRAIN_FRAMES[plan["train"]])
    seconds = n * EPOCHS * SECONDS_PER_ROW_EPOCH
    total += seconds
    rows.append({"run": P.run_name(plan["key"], FROZEN_R), "trains on": plan["train"],
                 "rows": f"{n:,}", "epochs": EPOCHS,
                 "estimate": f"{seconds / 60:.0f} min"})
display(pd.DataFrame(rows))

# Each model is scored on its validation set once more after being reloaded from
# disk. That pass is the only cost below that is not already inside the runtimes
# above; ~60 ms a row is the measured inference rate from 03_train.ipynb.
reload_seconds = sum(len(VAL_FRAMES[p["val"]]) for p in P.TRAINING_PLAN) * 0.06
total += reload_seconds

print(f"reference: the earlier run at r={FROZEN_R} took {float(frozen_row['runtime_seconds']):.0f}s "
      f"for {len(TRAIN_FRAMES['clean/train']):,} rows x {EPOCHS} epochs "
      f"({SECONDS_PER_ROW_EPOCH * 1000:.1f} ms per row-epoch)")
print(f"reload-and-verify passes                {reload_seconds / 60:.0f} min")
print(f"\nTOTAL ESTIMATE  {total / 60:.0f} min  ({total / 3600:.1f} hours)")
print("\nThe loop below resumes: a model whose adapter AND record both already exist is")
print("skipped, so a disconnect costs the model that was running, not the session.")

,run,trains on,rows,epochs,estimate
0,run_09_naive_r8,naive/train,"17,187",3,6 min
1,run_10_naive_sub_r8,naive_sub/train,"9,893",3,4 min


reference: the earlier run at r=8 took 221s for 9,893 rows x 3 epochs (7.5 ms per row-epoch)
reload-and-verify passes                7 min

TOTAL ESTIMATE  17 min  (0.3 hours)

The loop below resumes: a model whose adapter AND record both already exist is
skipped, so a disconnect costs the model that was running, not the session.


## 3 - Training the Two Models

Both models share a single configuration and differ only in training data. Every
hyperparameter is read from the frozen configuration record rather than from the constants
defined in `src/`, so that a value which has drifted since the freeze was written causes an
explicit failure rather than a silent discrepancy.

Each training run is subject to four validation steps before its result is accepted:

1. **No re-sampling of training rows.** The `train_rows` parameter is left unset so that
   `train_one_run()` uses the training frame exactly as provided, rather than drawing a fresh
   subsample. `naive_sub/train.csv` is itself the fixed subsample used for this comparison, and
   re-sampling within the training function would substitute a different, uncontrolled sample.
2. **Field-by-field comparison against the frozen configuration.** A run whose realised
   configuration differs from the frozen record raises an error before any result is written,
   ensuring that any observed difference between models reflects the training data rather than
   an unintended configuration difference.
3. **Reload validation.** Each adapter is reloaded from disk after training, and its evaluation
   score is required to match the recorded value. This step detects a specific failure mode in
   which a classification head is omitted from `modules_to_save`: training completes and the
   loss decreases normally, but the head is reinitialised randomly upon reloading, without
   raising an error.
4. **Incremental persistence.** Results are committed and mirrored to Drive before the next
   model begins training.

In [10]:
records, adapters, val_metrics = {}, {}, {}

for plan in P.TRAINING_PLAN:
    key = plan["key"]
    name = P.run_name(key, FROZEN_R)
    adapter_dir = RUNS_DIR / name
    record_path = METRICS_DIR / f"{name}.json"
    train_frame, val_frame = TRAIN_FRAMES[plan["train"]], VAL_FRAMES[plan["val"]]

    # Resume. If a previous session finished this run and both the adapter and its
    # record survived, retraining costs fifty minutes to produce the same weights -
    # and writes a second, differently-timed record of one run.
    if (adapter_dir / "adapter_model.safetensors").exists() and record_path.exists():
        records[key] = json.loads(record_path.read_text(encoding="utf-8"))
        adapters[key] = adapter_dir
        val_metrics[key] = records[key]["metrics"]
        print(f"=== {name}: already trained and recorded - skipping "
              f"(macro-F1 {records[key]['metrics']['f1_macro']:.4f}) ===\n")
        continue

    config = T.RunConfig(
        name=name,
        r=FROZEN_R,
        model_name=FREEZE["model"]["base_model"],
        epochs=EPOCHS,
        learning_rate=FREEZE["training"]["learning_rate"],
        batch_size=FREEZE["training"]["batch_size"],
        grad_accum=FREEZE["training"]["grad_accum"],
        warmup_ratio=FREEZE["training"]["warmup_ratio"],
        weight_decay=FREEZE["training"]["weight_decay"],
        train_seed=TRAIN_SEED,
        train_rows=None,                    # see point 1 above - not just a default
        trained_on=plan["train"],
        scored_on=plan["val"],
        subsample_seed=D.SUBSAMPLE_SEED if key == "naive_sub" else None,
        notes=f"protocol-models stage, frozen configuration. Trained on {plan['train']}, "
              f"best epoch chosen on {plan['val']}. Only the training protocol differs "
              "between the runs of this group.")

    print(f"=== {name}   {len(train_frame):,} rows -> selected on {plan['val']} ===")
    started = time.perf_counter()
    out = T.train_one_run(config, train_frame, val_frame, INTENTS, HARDWARE, adapter_dir)
    record = out["record"]
    reported = record["metrics"]["f1_macro"]
    print(f"    finished in {time.perf_counter() - started:.0f}s   macro-F1 {reported:.4f}"
          f"   (best epoch {record['config']['best_epoch']} of {EPOCHS})")

    P.assert_matches_freeze(record, FREEZE)
    print("    [PASS] every frozen field matches artifacts/config_freeze.json")

    # Free the trained model BEFORE loading another one. Two 1.7B models on one T4
    # is an out-of-memory error at the least useful possible moment.
    for field in ("model", "tokenizer", "val_encoded"):
        out.pop(field, None)
    gc.collect(); torch.cuda.empty_cache()

    # Reload from disk into a fresh model and require the recorded score back.
    model, tokenizer = T.load_adapter(adapter_dir, config.model_name, INTENTS,
                                      HARDWARE["precision"], device=HARDWARE["device"])
    encoded = T.encode_split(tokenizer, val_frame, INTENTS, HARDWARE["device"])
    logits = T.predict_logits(model, encoded, precision=HARDWARE["precision"]).numpy()
    metrics, rows = P.score_from_logits(logits, val_frame, INTENTS)
    if abs(metrics["f1_macro"] - reported) > 1e-6:
        raise AssertionError(
            f"{name}: the adapter on disk scores {metrics['f1_macro']:.6f} but the run "
            f"recorded {reported:.6f}. What was saved is not the epoch the metrics "
            "describe, so these weights and these numbers are about different models.")
    print(f"    [PASS] the adapter on disk reproduces {reported:.4f}")

    del model, tokenizer, encoded
    gc.collect(); torch.cuda.empty_cache()

    records[key], adapters[key], val_metrics[key] = record, adapter_dir, metrics
    P.save_val_outputs(name, logits, rows, METRICS_DIR)
    E.error_frame(val_frame, rows["predicted"], rows["confidence"]).to_csv(
        ERRORS_DIR / f"{name}_errors.csv", index=False)
    D.write_json(record, record_path)
    persist(f"{name} - macro-F1 {metrics['f1_macro']:.4f} on {plan['val']}")
    print()

print(f"{len(records)} of {len(P.TRAINING_PLAN)} models ready.")
print("clean/test and naive/test have not been read.")

=== run_09_naive_r8   17,187 rows -> selected on naive/val ===


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/3  loss 0.7027  val macro-F1 0.9962
  epoch 2/3  loss 0.0040  val macro-F1 0.9978
  epoch 3/3  loss 0.0002  val macro-F1 0.9975
    finished in 233s   macro-F1 0.9978   (best epoch 2 of 3)
    [PASS] every frozen field matches artifacts/config_freeze.json


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    [PASS] the adapter on disk reproduces 0.9978
  [git  ] committed (local to this runtime - Drive is what survives)
  [drive] mirrored, results/ on Drive now holds 77 files

2 of 2 models ready.
clean/test and naive/test have not been read.


## 4 - Summary: Data Varied, Configuration Held Constant

Two summary tables are presented. The first documents the training data used for each run; the
second documents that the training configuration was identical across runs. Establishing both
is necessary to attribute any difference in results to the training data alone: the absence of
a raised error is a weaker form of evidence than an explicit, inspectable record.

In [11]:
summary = pd.DataFrame([{
    "run": records[key]["name"],
    "trained on": records[key]["config"]["trained_on"],
    "train rows": f"{records[key]['config']['train_rows']:,}",
    "selected on": records[key]["config"]["scored_on"],
    "best epoch": records[key]["config"]["best_epoch"],
    "macro-F1 (val)": round(val_metrics[key]["f1_macro"], 4),
    "accuracy (val)": round(val_metrics[key]["accuracy"], 4),
    "train sha256": records[key]["config"]["train_sha256"][:16],
} for key in records])
display(summary)
summary.to_csv(METRICS_DIR / "protocol_models_val.csv", index=False)

# The naive model trains on 17,187 rows and the size control on 9,893. If the two
# shas matched, the "size control" would be a second copy of the naive run wearing
# a different name - and nothing else in this notebook would notice.
if len(records) == len(P.TRAINING_PLAN):
    shas = {records[k]["config"]["train_sha256"] for k in records}
    assert len(shas) == len(records), (
        "two runs share a train_sha256 - they trained on identical rows, so one of "
        "them is not the control it claims to be")
    print("\n[PASS] each run trained on a different set of rows, as it must")

print("\nNOTE these are VALIDATION scores, each on its own protocol's validation set.")
print("They are NOT comparable to each other: naive/val contains siblings of")
print("naive/train and clean/val does not, so the naive number is inflated by")
print("exactly the effect this project is measuring. The comparison needs the same")
print("test rows, and making it is 04_test.ipynb's entire purpose.")

,run,trained on,train rows,selected on,best epoch,macro-F1 (val),accuracy (val),train sha256
0,run_09_naive_r8,naive/train,"17,187",naive/val,3,0.9978,0.9978,0354eee23ff3b996
1,run_10_naive_sub_r8,naive_sub/train,"9,893",naive/val,2,0.9978,0.9978,d1a12470f92549af



[PASS] each run trained on a different set of rows, as it must

NOTE these are VALIDATION scores, each on its own protocol's validation set.
They are NOT comparable to each other: naive/val contains siblings of
naive/train and clean/val does not, so the naive number is inflated by
exactly the effect this project is measuring. The comparison needs the same
test rows, and making it is 04_test.ipynb's entire purpose.


In [12]:
for key in records:
    print("=" * 72)
    print(f"{records[key]['name']}  -  held fixed by artifacts/config_freeze.json:")
    frozen_check = P.matches_freeze(records[key], FREEZE)
    display(frozen_check)
    assert frozen_check["identical"].all(), P.freeze_violations(records[key], FREEZE)

    print(f"{records[key]['name']}  -  allowed to differ, and shown so:")
    display(P.differences_from_freeze(records[key], FREEZE))

print("[PASS] every frozen field identical on every run. Only the data moved.")

run_09_naive_r8  -  held fixed by artifacts/config_freeze.json:


,field,frozen,this run,identical
0,base_model,'Qwen/Qwen3-1.7B','Qwen/Qwen3-1.7B',True
1,task_type,'SEQ_CLS','SEQ_CLS',True
2,r,8,8,True
3,lora_alpha,16,16,True
4,lora_dropout,0.05,0.05,True
5,target_modules,"['q_proj', 'v_proj']","['q_proj', 'v_proj']",True
6,modules_to_save,['score'],['score'],True
7,learning_rate,0.0002,0.0002,True
8,epochs,3,3,True
9,batch_size,32,32,True


run_09_naive_r8  -  allowed to differ, and shown so:


,field,the frozen run,this run,differs
0,trained_on,'clean/train','naive/train',True
1,scored_on,'clean/val','naive/val',True
2,train_rows,9893,17187,True
3,eval_rows,2120,3683,True
4,train_sha256,'6d674bcb7d8fd254705cf847ebeefae9e3f9e356f385b...,'0354eee23ff3b99681ffecf4eefe46d5037953ecafa77...,True
5,best_epoch,-,3,-
6,subsample_seed,-,None,-


run_10_naive_sub_r8  -  held fixed by artifacts/config_freeze.json:


,field,frozen,this run,identical
0,base_model,'Qwen/Qwen3-1.7B','Qwen/Qwen3-1.7B',True
1,task_type,'SEQ_CLS','SEQ_CLS',True
2,r,8,8,True
3,lora_alpha,16,16,True
4,lora_dropout,0.05,0.05,True
5,target_modules,"['q_proj', 'v_proj']","['q_proj', 'v_proj']",True
6,modules_to_save,['score'],['score'],True
7,learning_rate,0.0002,0.0002,True
8,epochs,3,3,True
9,batch_size,32,32,True


run_10_naive_sub_r8  -  allowed to differ, and shown so:


,field,the frozen run,this run,differs
0,trained_on,'clean/train','naive_sub/train',True
1,scored_on,'clean/val','naive/val',True
2,train_rows,9893,9893,False
3,eval_rows,2120,3683,True
4,train_sha256,'6d674bcb7d8fd254705cf847ebeefae9e3f9e356f385b...,'d1a12470f92549af7d88c58030a7ec68f879e1b581b2f...,True
5,best_epoch,-,2,-
6,subsample_seed,-,42,-


[PASS] every frozen field identical on every run. Only the data moved.


## 5 - Verification and Session Closure

The absence of an error from `save_pretrained` does not by itself confirm that a file has been
written to persistent storage: a Drive mount can become unavailable mid-session, in which case
writes succeed locally but are lost when the runtime is destroyed. The saved files are
therefore explicitly verified before the runtime is closed.

The final check in this section is not a condition for this notebook's completion; it confirms
readiness for the evaluation notebook that follows.

In [13]:
print("adapters:")
for key, adapter in adapters.items():
    weights = Path(adapter) / "adapter_model.safetensors"
    size = weights.stat().st_size / 1e6 if weights.exists() else 0
    print(f"  [{'OK  ' if size else 'GONE'}]  {records[key]['name']:22s} "
          f"{size:7.1f} MB  {adapter}")

print("\nrow-level outputs (later analysis reads these, on CPU, without a GPU or the test set):")
for key in records:
    name = records[key]["name"]
    for path in (METRICS_DIR / f"val_logits_{name}.npy",
                 METRICS_DIR / f"val_predictions_{name}.csv",
                 METRICS_DIR / f"{name}.json",
                 ERRORS_DIR / f"{name}_errors.csv"):
        mark = "OK  " if path.exists() else "MISS"
        size = f"{path.stat().st_size / 1024:8.1f} KB" if path.exists() else " " * 11
        print(f"  [{mark}]  {path.relative_to(REPO_ROOT).as_posix():56s}{size}")

persist("step 18a complete - the protocol models are trained")



adapters:
  [OK  ]  run_09_naive_r8            6.7 MB  /content/drive/MyDrive/support-triage/runs/run_09_naive_r8
  [OK  ]  run_10_naive_sub_r8        6.7 MB  /content/drive/MyDrive/support-triage/runs/run_10_naive_sub_r8

row-level outputs (later analysis reads these, on CPU, without a GPU or the test set):
  [OK  ]  results/metrics/val_logits_run_09_naive_r8.npy             388.6 KB
  [OK  ]  results/metrics/val_predictions_run_09_naive_r8.csv        462.0 KB
  [OK  ]  results/metrics/run_09_naive_r8.json                        24.9 KB
  [OK  ]  results/errors/run_09_naive_r8_errors.csv                    0.8 KB
  [OK  ]  results/metrics/val_logits_run_10_naive_sub_r8.npy         388.6 KB
  [OK  ]  results/metrics/val_predictions_run_10_naive_sub_r8.csv    462.0 KB
  [OK  ]  results/metrics/run_10_naive_sub_r8.json                    17.4 KB
  [OK  ]  results/errors/run_10_naive_sub_r8_errors.csv                0.9 KB
  [git  ] committed (local to this runtime - Drive is what survive

In [14]:
# Not a condition of this notebook - a favour to the next one. 04_test scores the
# clean model from the frozen run in 03_train.ipynb rather than retraining it, and
# discovering in THAT session that the adapter is gone would cost fifty minutes there.
clean_adapter = RUNS_DIR / FREEZE["frozen_from_run"]
weights = clean_adapter / "adapter_model.safetensors"
if weights.exists():
    print(f"[OK  ] {FREEZE['frozen_from_run']} is on Drive "
          f"({weights.stat().st_size / 1e6:.1f} MB)")
    print("       04_test can score the clean model without retraining it.")
else:
    print("!" * 70)
    print(f"{FREEZE['frozen_from_run']} is NOT at {clean_adapter}.")
    print("The freeze names it as the run the configuration was frozen from, and")
    print("04_test scores the clean model from it. If it really is gone, that")
    print("notebook has to retrain the clean model and needs ~50 minutes more.")
    print("Decide that there, with the freeze in front of you - not here.")
    print("!" * 70)

[OK  ] run_04_lora_r8 is on Drive (6.7 MB)
       04_test can score the clean model without retraining it.


In [15]:
# Only after the cell above shows everything committed or mirrored. Unassigning
# destroys the machine and everything on its local disk with it.
print("Everything is written. Closing the runtime.")

if IN_COLAB:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print(f"could not unassign automatically ({type(exc).__name__}) - "
              "use Runtime > Disconnect and delete runtime")

Everything is written. Closing the runtime.


### Summary and Next Steps

Completion of this notebook requires that the naive-split model and its size-matched control
(saved as `run_09_naive_r16` and `run_10_naive_sub_r16`, the adapter and metrics filenames this
notebook writes to disk) have been trained and verified against the frozen configuration, with
their outputs mirrored to Drive and committed locally. Following the session, the results are
restored from the Drive mirror into the local repository and pushed from there, since the
Colab runtime holds no push credentials.

The next stage of the protocol is `04_test.ipynb`, run in a separate session, which performs
the single evaluation pass on the held-out test set. That notebook requires three trained
adapters: the two produced here, together with `run_05_lora_r16` from `03_train.ipynb`, which
provides the clean-split model. Neither `clean/test` nor `naive/test` was accessed at any point
in this notebook.